###Part A — Data Profiling & Operational Audit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.7 MB/s eta 0:00:00


In [ ]:
#1.Load dataset
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/UrbanEats/urbaneats_delivery_orders.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Shape: (150, 11)

Columns: ['order_id', 'order_date', 'restaurant_name', 'delivery_zone', 'order_value', 'delivery_time_mins', 'rider_rating', 'order_status', 'payment_method', 'discount_applied', 'customer_complaints']

First 5 rows:


,order_id,order_date,restaurant_name,delivery_zone,order_value,delivery_time_mins,rider_rating,order_status,payment_method,discount_applied,customer_complaints
0,ORD00001,9/25/2024,Pizza Palace,North,1705,NaN,3.0,Delayed,Cash,12,0
1,ORD00002,3/11/2024,Pizza Palace,East,807,NaN,4.2,Cancelled,Card,12,3
2,ORD00003,12/11/2024,Spice Garden,South,466,NaN,3.6,Delayed,Card,1,3
3,ORD00004,7/10/2024,Wrap & Roll,East,675,NaN,3.6,Delivered,UPI,3,0
4,ORD00005,3/15/2024,Wrap & Roll,North,349,NaN,2.0,Refunded,Cash,20,0


In [ ]:
#2.ydata_profiling
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="UrbanEats Operations Audit", explorative=True)
profile.to_file("/content/drive/MyDrive/UrbanEats/urbaneats_profile.html")
print("Done!")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 11/11 [00:00<00:00, 88.61it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Done!


In [ ]:
total = len(df)

# 1. Missing delivery_time_mins by order_status
missing_time = df["delivery_time_mins"].isna().sum()
missing_time_by_status = df[df["delivery_time_mins"].isna()]["order_status"].value_counts()

# 2. Missing rider_rating by order_status
missing_rating = df["rider_rating"].isna().sum()
missing_rating_by_status = df[df["rider_rating"].isna()]["order_status"].value_counts()

# 3. Order status distribution
status_dist = df["order_status"].value_counts()
status_pct = df["order_status"].value_counts(normalize=True) * 100

# 4. Order value check
zero_orders = (df["order_value"] == 0).sum()
negative_orders = (df["order_value"] < 0).sum()

print(f"Total rows: {total}")
print(f"\nMissing delivery_time_mins: {missing_time}")
print(f"By order_status:\n{missing_time_by_status}")
print(f"\nMissing rider_rating: {missing_rating}")
print(f"By order_status:\n{missing_rating_by_status}")
print(f"\nOrder status distribution:\n{status_dist}")
print(f"\nPercentage:\n{status_pct.round(2)}")
print(f"\nZero value orders: {zero_orders}")
print(f"Negative value orders: {negative_orders}")

Total rows: 150

Missing delivery_time_mins: 6
By order_status:
order_status
Delayed      2
Delivered    2
Cancelled    1
Refunded     1
Name: count, dtype: int64

Missing rider_rating: 6
By order_status:
order_status
Delivered    3
Cancelled    2
Refunded     1
Name: count, dtype: int64

Order status distribution:
order_status
Delivered    44
Refunded     37
Cancelled    36
Delayed      33
Name: count, dtype: int64

Percentage:
order_status
Delivered    29.33
Refunded     24.67
Cancelled    24.00
Delayed      22.00
Name: proportion, dtype: float64

Zero value orders: 0
Negative value orders: 0


#3. Audit Findings

###  Missing delivery_time_mins
6 orders are missing delivery_time_mins, spread across all status types:
- Delayed: 2, Delivered: 2, Cancelled: 1, Refunded: 1
- No strong concentration in one status — missing values are scattered.

###  Missing rider_rating
6 orders are missing rider_rating:
- Delivered: 3, Cancelled: 2, Refunded: 1
- Slight concentration in Delivered orders — unexpected, as completed orders should always have a rating.

###  Order Status Distribution
- Delivered: 44 (29.33%)
- Refunded: 37 (24.67%)
- Cancelled: 36 (24.00%)
- Delayed: 33 (22.00%)
- Combined Cancelled + Refunded = 48.67% — nearly half of all orders did not complete successfully.

###  Order Value
- No zero or negative order values found — order_value is clean.

#4. VP Summary
delivery_time_mins is the most critical missing field for delivery time analysis
because it is the only direct measure of operational speed — without it,
we cannot calculate average delivery times or identify which zones and
restaurants are causing delays for 6 orders across all status types.

#Part B — Great Expectations Quality Gate

In [ ]:
!pip install great-expectations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
import great_expectations as gx

context = gx.get_context()
data_source = context.data_sources.add_pandas("urbaneats_datasource")
data_asset = data_source.add_dataframe_asset("urbaneats")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
suite = context.suites.add(gx.ExpectationSuite(name="urbaneats_suite"))
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(name="urbaneats_val", data=batch_definition, suite=suite)
)
print("Setup done!")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpa9q4txmh' for ephemeral docs site


Setup done!


In [ ]:
# 5. order_id must not be null — duplicate/missing orders inflate all volume metrics
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id")
)

# 5. order_id must be unique — duplicate orders inflate all volume metrics
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column="order_id")
)

# 6. delivery_time_mins must be between 10 and 120 — outside range = data entry errors
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="delivery_time_mins", min_value=10, max_value=120
    )
)

# 7. rider_rating must be between 1.0 and 5.0
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="rider_rating", min_value=1.0, max_value=5.0
    )
)

# 8. order_status must be one of the 4 valid values
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="order_status",
        value_set=["Delivered", "Cancelled", "Delayed", "Refunded"]
    )
)

# 9. order_value must be between 50 and 5000
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_value", min_value=50, max_value=5000
    )
)

print("6 expectations added!")

6 expectations added!


In [ ]:
results = validation_definition.run(batch_parameters={"dataframe": df})

# Summary
stats = results.to_json_dict()["statistics"]
print(f"Evaluated: {stats['evaluated_expectations']}")
print(f"Passed: {stats['successful_expectations']}")
print(f"Failed: {stats['unsuccessful_expectations']}")
print(f"Success %: {stats['success_percent']}")

Calculating Metrics:   0%|          | 0/44 [00:00<?, ?it/s]

Evaluated: 6
Passed: 6
Failed: 0
Success %: 100.0


In [ ]:
import json

results_dict = results.to_json_dict()

html_content = f"""
<html><head><title>UrbanEats GX Report</title></head>
<body>
<h1>Great Expectations Validation Report — UrbanEats</h1>
<h2>Success: {results_dict['success']}</h2>
<h3>Statistics</h3>
<p>Evaluated: {results_dict['statistics']['evaluated_expectations']}</p>
<p>Passed: {results_dict['statistics']['successful_expectations']}</p>
<p>Failed: {results_dict['statistics']['unsuccessful_expectations']}</p>
<p>Success %: {results_dict['statistics']['success_percent']}</p>
<pre>{json.dumps(results_dict, indent=2)}</pre>
</body></html>
"""

with open("/content/drive/MyDrive/UrbanEats/urbaneats_gx_report.html", "w") as f:
    f.write(html_content)

print("GX report saved!")

GX report saved!


## Part B — GX Validation Summary (For VP's EA)

- **What passed:** All 6 data quality checks passed — order IDs are unique
  and complete, rider ratings are within valid range (1–5), order status
  values are all recognised, and order values fall within expected limits.

- **What failed:** None — the dataset is structurally clean and ready
  for analysis.

- **What it means for this week's analysis:** The data can be used
  immediately without any structural fixes. However, 6 missing
  delivery_time_mins and 6 missing rider_rating values were noted in
  Part A — these will be handled during modelling but do not block
  the analysis.